# Library of 1.2 million sequences 

## F-centered sequences

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import least_squares

# ========== Configuration ==========
input_path = "./5letter_library/"
test_file = "F_rows.csv"

middle_position = 5
aa_list = list("LRGNE")

# Scaling
SCALE_FACTOR = 1/(10**-6)**(1/8)

# Threshold parameters
BASE_THRESHOLD = 40
MIN_THRESHOLD = 9
R_WEIGHT = 15
K_WEIGHT = 10


# ========== Sigmoid ==========
def sigmoid(Q):
    if Q <= 0:
        return 0.0
    return Q / (1 + Q)


# ========== Model Prediction ==========
def model_prediction(seq, params):
    product = 1.0
    aa_param_map = dict(zip(aa_list, params))

    for index, aa in enumerate(seq, start=1):
        if index == middle_position or aa not in aa_param_map:
            continue
        product *= aa_param_map[aa]

    return sigmoid(product)


# ========== Threshold ==========
def compute_dynamic_threshold(seq):
    num_r = seq.count('R')
    num_k = seq.count('K')
    return max(BASE_THRESHOLD - R_WEIGHT*num_r - K_WEIGHT*num_k, MIN_THRESHOLD)


# ========== Residuals ==========
def residuals(params, sequences, labels):
    return [
        label - model_prediction(seq, params)
        for seq, label in zip(sequences, labels)
    ]


def squared_residuals(params, sequences, labels):
    return [r**2 for r in residuals(params, sequences, labels)]


# ========== Load Data ==========
df = pd.read_csv(f"{input_path}{test_file}", header=None, sep=r"\s+")

sequences = df[0].tolist()
values = df[3].tolist()


# ========== Label Assignment ==========
labels = []

for seq, val in zip(sequences, values):

    if 'R' not in seq and 'K' not in seq:
        label = 1 if val >= BASE_THRESHOLD else 0
    else:
        threshold = compute_dynamic_threshold(seq)
        label = 1 if val >= threshold else 0

    labels.append(label)


# ========== Initial Guesses ==========
initial_guess_dict = {
    'L': 0.41,
    'R': 0.27,
    'E': 0.11,
    'G': 0.15,
    'N': 0.12
}

lower_bounds = []
upper_bounds = []
initial_guess = []

for aa in aa_list:

    if aa == 'L':
        lb, ub = 0.1, 3.0
    else:
        lb, ub = 0.01, 0.5

    lower_bounds.append(lb * SCALE_FACTOR)
    upper_bounds.append(ub * SCALE_FACTOR)
    initial_guess.append(initial_guess_dict[aa] * SCALE_FACTOR)


# ========== Optimization ==========
result = least_squares(
    squared_residuals,
    initial_guess,
    bounds=(lower_bounds, upper_bounds),
    args=(sequences, labels),
    method='trf'
)


# ========== Results ==========
output = {}

if result.success:

    params = result.x

    for aa, val in zip(aa_list, params):
        output[aa] = val

    scores = [model_prediction(seq, params) for seq in sequences]
    preds = [1 if s >= 0.5 else 0 for s in scores]

    TP = sum(p==1 and l==1 for p,l in zip(preds,labels))
    TN = sum(p==0 and l==0 for p,l in zip(preds,labels))
    FP = sum(p==1 and l==0 for p,l in zip(preds,labels))
    FN = sum(p==0 and l==1 for p,l in zip(preds,labels))

    accuracy = (TP+TN)/len(labels)*100

    output.update({
        "Accuracy": accuracy,
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN
    })

else:
    output["Error"] = result.message


# ========== Print ==========
final_df = pd.DataFrame([output])
print(final_df)

/home/fidha/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/fidha/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


          L         R         G         N         E  Accuracy      TP      TN  \
0  2.261206  1.481934  0.797487  0.640801  0.575959  87.53792  166432  175513   

      FP     FN  
0  25308  23372  


## W-centered sequences

In [9]:
import pandas as pd
import numpy as np
from scipy.optimize import least_squares

# ========== Configuration ==========
input_path = "./5letter_library/"
test_file = "W_rows.csv"

middle_position = 5
aa_list = list("LRGNE")

# Scaling
SCALE_FACTOR = 1/(10**-6)**(1/8)

# Threshold parameters
BASE_THRESHOLD = 40
MIN_THRESHOLD = 9
R_WEIGHT = 15
K_WEIGHT = 10


# ========== Sigmoid ==========
def sigmoid(Q):
    if Q <= 0:
        return 0.0
    return Q / (1 + Q)


# ========== Model Prediction ==========
def model_prediction(seq, params):
    product = 1.0
    aa_param_map = dict(zip(aa_list, params))

    for index, aa in enumerate(seq, start=1):
        if index == middle_position or aa not in aa_param_map:
            continue
        product *= aa_param_map[aa]

    return sigmoid(product)


# ========== Threshold ==========
def compute_dynamic_threshold(seq):
    num_r = seq.count('R')
    num_k = seq.count('K')
    return max(BASE_THRESHOLD - R_WEIGHT*num_r - K_WEIGHT*num_k, MIN_THRESHOLD)


# ========== Residuals ==========
def residuals(params, sequences, labels):
    return [
        label - model_prediction(seq, params)
        for seq, label in zip(sequences, labels)
    ]


def squared_residuals(params, sequences, labels):
    return [r**2 for r in residuals(params, sequences, labels)]


# ========== Load Data ==========
df = pd.read_csv(f"{input_path}{test_file}", header=None, sep=r"\s+")

sequences = df[0].tolist()
values = df[3].tolist()


# ========== Label Assignment ==========
labels = []

for seq, val in zip(sequences, values):

    if 'R' not in seq and 'K' not in seq:
        label = 1 if val >= BASE_THRESHOLD else 0
    else:
        threshold = compute_dynamic_threshold(seq)
        label = 1 if val >= threshold else 0

    labels.append(label)


# ========== Initial Guesses (SCALED) ==========
initial_guess_dict = {
    'L': 0.42,     # L gets a higher starting value
    'R': 0.27,
    'E': 0.09,
    'G': 0.13,
    'N': 0.10
}

lower_bounds = []
upper_bounds = []
initial_guess = []

for aa in aa_list:

    if aa == 'L':
        lb, ub = 0.1, 3.0
    else:
        lb, ub = 0.01, 0.5

    lower_bounds.append(lb * SCALE_FACTOR)
    upper_bounds.append(ub * SCALE_FACTOR)
    initial_guess.append(initial_guess_dict[aa] * SCALE_FACTOR)


# ========== Optimization ==========
result = least_squares(
    squared_residuals,
    initial_guess,
    bounds=(lower_bounds, upper_bounds),
    args=(sequences, labels),
    method='trf'
)


# ========== Results ==========
output = {}

if result.success:

    params = result.x

    for aa, val in zip(aa_list, params):
        output[aa] = val

    scores = [model_prediction(seq, params) for seq in sequences]
    preds = [1 if s >= 0.5 else 0 for s in scores]

    TP = sum(p==1 and l==1 for p,l in zip(preds,labels))
    TN = sum(p==0 and l==0 for p,l in zip(preds,labels))
    FP = sum(p==1 and l==0 for p,l in zip(preds,labels))
    FN = sum(p==0 and l==1 for p,l in zip(preds,labels))

    accuracy = (TP+TN)/len(labels)*100

    output.update({
        "Accuracy": accuracy,
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN
    })

else:
    output["Error"] = result.message


# ========== Print ==========
final_df = pd.DataFrame([output])
print(final_df)

         L        R         G         N        E   Accuracy      TP      TN  \
0  2.37161  1.49852  0.744095  0.593974  0.52908  90.133504  143525  208559   

      FP     FN  
0  16407  22134  


## Y-centered sequences

In [10]:
import pandas as pd
import numpy as np
from scipy.optimize import least_squares

# ========== Configuration ==========
input_path = "./5letter_library/"
test_file = "Y_rows.csv"

middle_position = 5
aa_list = list("LRGNE")

# Scaling
SCALE_FACTOR = 1/(10**-1)**(1/8)   # (10^6)^(1/8)

# Threshold parameters
BASE_THRESHOLD = 40
MIN_THRESHOLD = 9
R_WEIGHT = 15
K_WEIGHT = 10


# ========== Sigmoid ==========
def sigmoid(Q):
    if Q <= 0:
        return 0.0
    return Q / (1 + Q)


# ========== Model Prediction ==========
def model_prediction(seq, params):
    product = 1.0
    aa_param_map = dict(zip(aa_list, params))

    for index, aa in enumerate(seq, start=1):
        if index == middle_position or aa not in aa_param_map:
            continue
        product *= aa_param_map[aa]

    return sigmoid(product)


# ========== Threshold ==========
def compute_dynamic_threshold(seq):
    num_r = seq.count('R')
    num_k = seq.count('K')
    return max(BASE_THRESHOLD - R_WEIGHT*num_r - K_WEIGHT*num_k, MIN_THRESHOLD)


# ========== Residuals ==========
def residuals(params, sequences, labels):
    return [
        label - model_prediction(seq, params)
        for seq, label in zip(sequences, labels)
    ]


def squared_residuals(params, sequences, labels):
    return [r**2 for r in residuals(params, sequences, labels)]


# ========== Load Data ==========
df = pd.read_csv(f"{input_path}{test_file}", header=None, sep=r"\s+")

sequences = df[0].tolist()
values = df[3].tolist()


# ========== Label Assignment ==========
labels = []

for seq, val in zip(sequences, values):

    if 'R' not in seq and 'K' not in seq:
        label = 1 if val >= BASE_THRESHOLD else 0
    else:
        threshold = compute_dynamic_threshold(seq)
        label = 1 if val >= threshold else 0

    labels.append(label)


# ========== Initial Guesses (SCALED) ==========
initial_guess_dict = {
    'L': 0.70,     # L gets a higher starting value
    'R': 0.50,
    'E': 0.22,
    'G': 0.28,
    'N': 0.23
}

lower_bounds = []
upper_bounds = []
initial_guess = []

for aa in aa_list:

    if aa == 'L':
        lb, ub = 0.1, 3.0
    else:
        lb, ub = 0.01, 1.0

    lower_bounds.append(lb * SCALE_FACTOR)
    upper_bounds.append(ub * SCALE_FACTOR)
    initial_guess.append(initial_guess_dict[aa] * SCALE_FACTOR)


# ========== Optimization ==========
result = least_squares(
    squared_residuals,
    initial_guess,
    bounds=(lower_bounds, upper_bounds),
    args=(sequences, labels),
    method='trf'
)


# ========== Results ==========
output = {}

if result.success:

    params = result.x

    for aa, val in zip(aa_list, params):
        output[aa] = val

    scores = [model_prediction(seq, params) for seq in sequences]
    preds = [1 if s >= 0.5 else 0 for s in scores]

    TP = sum(p==1 and l==1 for p,l in zip(preds,labels))
    TN = sum(p==0 and l==0 for p,l in zip(preds,labels))
    FP = sum(p==1 and l==0 for p,l in zip(preds,labels))
    FN = sum(p==0 and l==1 for p,l in zip(preds,labels))

    accuracy = (TP+TN)/len(labels)*100

    output.update({
        "Accuracy": accuracy,
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN
    })

else:
    output["Error"] = result.message


# ========== Print ==========
final_df = pd.DataFrame([output])
print(final_df)

          L         R        G         N         E   Accuracy    TP      TN  \
0  1.353762  0.958998  0.51083  0.461751  0.428472  98.783744  1660  384214   

     FP    FN  
0  1167  3584  


# IDR dataset

## F-centered sequences 

### Training

In [17]:
import pandas as pd
import numpy as np
from scipy.optimize import least_squares

# ================= Configuration =================
file_path = "./IDR_library/F_rows_train.csv"
middle_position = 5

# -------- Scale Factor --------
SCALE_FACTOR = 1/(10**-6)**(1/8)

# Thresholding parameters
BASE_THRESHOLD = 40
MIN_THRESHOLD = 9
R_WEIGHT = 15
K_WEIGHT = 10

# Amino acids and groups
aa_list = list("ACDEFGHIKLMNPQRSTVWY")

aa_groups = {
    "group1": list("VLIMFWY"),
    "group2": list("GPCA"),
    "group3": list("STNQH"),
    "group4": list("KR"),
    "group5": list("DE"),
}

aa_to_group = {aa: g for g, aas in aa_groups.items() for aa in aas}


# ================= Sigmoid =================
def sigmoid(Q):
    if Q <= 0:
        return 0.0
    return Q / (1 + Q)


# ================= Model =================
def model_prediction(seq, params):
    product = 1.0
    param_map = {aa: params[i] for i, aa in enumerate(aa_list)}

    for index, aa in enumerate(seq, start=1):
        if index == middle_position or aa not in aa_list:
            continue

        product *= max(1e-12, param_map[aa])

    return sigmoid(product)


# ================= Residuals =================
def residuals(params, sequences, labels):
    return [
        label - model_prediction(seq, params)
        for seq, label in zip(sequences, labels)
    ]


def sum_squared_residuals(params, sequences, labels):
    r = residuals(params, sequences, labels)
    return [x**2 for x in r]


# ================= Threshold =================
def compute_dynamic_threshold(seq):
    num_r = seq.count('R')
    num_k = seq.count('K')
    return max(BASE_THRESHOLD - R_WEIGHT * num_r - K_WEIGHT * num_k, MIN_THRESHOLD)


# ================= Bounds & Initial Guess (SCALED) =================
def get_bounds_and_guess():

    lower_bounds = []
    upper_bounds = []
    initial_guess = []

    for aa in aa_list:

        group = aa_to_group.get(aa, "group5")

        if group == "group1":
            lb, ub, ig = 0.1, 5.0, 1.87
        elif group == "group2":
            lb, ub, ig = 0.01, 2.0, 0.28
        elif group == "group3":
            lb, ub, ig = 0.01, 1.5, 0.21
        elif group == "group4":
            lb, ub, ig = 0.01, 1.0, 0.32
        else:
            lb, ub, ig = 0.01, 1.0, 0.19

        lower_bounds.append(lb * SCALE_FACTOR)
        upper_bounds.append(ub * SCALE_FACTOR)
        initial_guess.append(ig * SCALE_FACTOR)

    return lower_bounds, upper_bounds, initial_guess


# ================= Load Data =================
df = pd.read_csv(file_path, header=None, sep=r"\s+")
df = df[~df[0].str.contains("X")]

sequences = df[0].tolist()
values = df[3].tolist()


# ================= Label Assignment =================
labels = []

for seq, val in zip(sequences, values):

    if 'R' not in seq and 'K' not in seq:
        label = 1 if val >= BASE_THRESHOLD else 0
    else:
        threshold = compute_dynamic_threshold(seq)
        label = 1 if val >= threshold else 0

    labels.append(label)


# ================= Optimization =================
lower_bounds, upper_bounds, initial_guess = get_bounds_and_guess()

result = least_squares(
    sum_squared_residuals,
    initial_guess,
    bounds=(lower_bounds, upper_bounds),
    args=(sequences, labels),
    method='trf'
)


# ================= Output =================
results_dict = {"AA": aa_list + ["Accuracy", "TP", "TN", "FP", "FN"]}

if result.success:

    params = result.x

    raw_scores = [
        model_prediction(seq, params)
        for seq in sequences
    ]

    preds = [1 if score >= 0.5 else 0 for score in raw_scores]

    TP = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 1)
    TN = sum(1 for p, l in zip(preds, labels) if p == 0 and l == 0)
    FP = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 0)
    FN = sum(1 for p, l in zip(preds, labels) if p == 0 and l == 1)

    accuracy = (TP + TN) / len(labels) * 100

    column_data = list(params) + [accuracy, TP, TN, FP, FN]

else:
    column_data = [np.nan] * (len(aa_list) + 5)


results_dict["Run_full"] = column_data
results_df = pd.DataFrame(results_dict)

print(results_df)

          AA     Run_full
0          A     1.075807
1          C     0.751508
2          D     0.409886
3          E     0.398312
4          F     8.160393
5          G     0.635416
6          H     0.784669
7          I     3.149122
8          K     1.100925
9          L     3.604654
10         M     2.865332
11         N     0.480348
12         P     1.769498
13         Q     0.493350
14         R     1.658523
15         S     0.930168
16         T     1.241720
17         V     2.035559
18         W     5.906307
19         Y     1.880200
20  Accuracy    91.617379
21        TP  4892.000000
22        TN  3163.000000
23        FP   380.000000
24        FN   357.000000


### Testing

In [7]:
import pandas as pd
import numpy as np

# ================= Test File =================
test_file_path = "./IDR_library/F_rows_test.csv"

# ================= Load Test Data =================
df_test = pd.read_csv(test_file_path, header=None, sep=r"\s+")
df_test = df_test[~df_test[0].str.contains("X")]

test_sequences = df_test[0].tolist()
test_values = df_test[3].tolist()

# ================= Create Test Labels  =================
test_labels = []

for seq, val in zip(test_sequences, test_values):
    if 'R' not in seq and 'K' not in seq:
        label = 1 if val >= BASE_THRESHOLD else 0
    else:
        threshold = compute_dynamic_threshold(seq)
        label = 1 if val >= threshold else 0
    test_labels.append(label)

# ================= Use Trained Parameters =================
# Make sure this comes from training block
trained_params = params   # result.x from training

# ================= Predict on Test =================
test_raw_scores = [
    model_prediction(seq, trained_params)
    for seq in test_sequences
]

test_preds = [1 if score >= 0.5 else 0 for score in test_raw_scores]

# ================= Confusion Matrix =================
TP = sum(1 for p, l in zip(test_preds, test_labels) if p == 1 and l == 1)
TN = sum(1 for p, l in zip(test_preds, test_labels) if p == 0 and l == 0)
FP = sum(1 for p, l in zip(test_preds, test_labels) if p == 1 and l == 0)
FN = sum(1 for p, l in zip(test_preds, test_labels) if p == 0 and l == 1)

accuracy = (TP + TN) / len(test_labels) * 100

# ================= Print Results =================
print("\n===== TEST SET RESULTS =====")
print("Accuracy:", round(accuracy, 3), "%")
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)


===== TEST SET RESULTS =====
Accuracy: 91.182 %
TP: 1223
TN: 783
FP: 94
FN: 100


## W-centered sequences

### Training

In [30]:
import pandas as pd
import numpy as np
from scipy.optimize import least_squares

# ================= Configuration =================
file_path = "./W_rows_train.csv"
middle_position = 5

# -------- Scale Factor --------
SCALE_FACTOR = 1/(10**-6)**(1/8)

# Thresholding parameters
BASE_THRESHOLD = 40
MIN_THRESHOLD = 9
R_WEIGHT = 15
K_WEIGHT = 10

# Amino acids and groups
aa_list = list("ACDEFGHIKLMNPQRSTVWY")

aa_groups = {
    "group1": list("VLIMFWY"),
    "group2": list("GPCA"),
    "group3": list("STNQH"),
    "group4": list("KR"),
    "group5": list("DE"),
}

aa_to_group = {aa: g for g, aas in aa_groups.items() for aa in aas}


# ================= Sigmoid =================
def sigmoid(Q):
    if Q <= 0:
        return 0.0
    return Q / (1 + Q)


# ================= Model =================
def model_prediction(seq, params):
    product = 1.0
    param_map = {aa: params[i] for i, aa in enumerate(aa_list)}

    for index, aa in enumerate(seq, start=1):
        if index == middle_position or aa not in aa_list:
            continue

        product *= max(1e-12, param_map[aa])

    return sigmoid(product)


# ================= Residuals =================
def residuals(params, sequences, labels):
    return [
        label - model_prediction(seq, params)
        for seq, label in zip(sequences, labels)
    ]


def sum_squared_residuals(params, sequences, labels):
    r = residuals(params, sequences, labels)
    return [x**2 for x in r]


# ================= Threshold  =================
def compute_dynamic_threshold(seq):
    num_r = seq.count('R')
    num_k = seq.count('K')
    return max(BASE_THRESHOLD - R_WEIGHT * num_r - K_WEIGHT * num_k, MIN_THRESHOLD)


# ================= Bounds & Initial Guess (SCALED) =================
def get_bounds_and_guess():

    lower_bounds = []
    upper_bounds = []
    initial_guess = []

    for aa in aa_list:

        group = aa_to_group.get(aa, "group5")

        if group == "group1":
            lb, ub, ig = 0.1, 5.0, 1.87
        elif group == "group2":
            lb, ub, ig = 0.01, 2.0, 0.28
        elif group == "group3":
            lb, ub, ig = 0.01, 1.5, 0.21
        elif group == "group4":
            lb, ub, ig = 0.01, 1.0, 0.32
        else:
            lb, ub, ig = 0.01, 1.0, 0.19

        lower_bounds.append(lb * SCALE_FACTOR)
        upper_bounds.append(ub * SCALE_FACTOR)
        initial_guess.append(ig * SCALE_FACTOR)

    return lower_bounds, upper_bounds, initial_guess


# ================= Load Data =================
df = pd.read_csv(file_path, header=None, sep=r"\s+")
df = df[~df[0].str.contains("X")]

sequences = df[0].tolist()
values = df[3].tolist()


# ================= Label Assignment =================
labels = []

for seq, val in zip(sequences, values):

    if 'R' not in seq and 'K' not in seq:
        label = 1 if val >= BASE_THRESHOLD else 0
    else:
        threshold = compute_dynamic_threshold(seq)
        label = 1 if val >= threshold else 0

    labels.append(label)


# ================= Optimization =================
lower_bounds, upper_bounds, initial_guess = get_bounds_and_guess()

result = least_squares(
    sum_squared_residuals,
    initial_guess,
    bounds=(lower_bounds, upper_bounds),
    args=(sequences, labels),
    method='trf'
)


# ================= Output =================
results_dict = {"AA": aa_list + ["Accuracy", "TP", "TN", "FP", "FN"]}

if result.success:

    params = result.x

    raw_scores = [
        model_prediction(seq, params)
        for seq in sequences
    ]

    preds = [1 if score >= 0.5 else 0 for score in raw_scores]

    TP = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 1)
    TN = sum(1 for p, l in zip(preds, labels) if p == 0 and l == 0)
    FP = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 0)
    FN = sum(1 for p, l in zip(preds, labels) if p == 0 and l == 1)

    accuracy = (TP + TN) / len(labels) * 100

    column_data = list(params) + [accuracy, TP, TN, FP, FN]

else:
    column_data = [np.nan] * (len(aa_list) + 5)


results_dict["Run_full"] = column_data
results_df = pd.DataFrame(results_dict)

print(results_df)

          AA     Run_full
0          A     1.020644
1          C     0.670821
2          D     0.392559
3          E     0.400095
4          F    12.431560
5          G     0.595651
6          H     0.719176
7          I     2.925139
8          K     1.011214
9          L     4.086136
10         M     2.850648
11         N     0.473254
12         P     1.609327
13         Q     0.476733
14         R     1.449318
15         S     0.905316
16         T     1.125488
17         V     2.041731
18         W    11.557157
19         Y     1.923680
20  Accuracy    91.359649
21        TP  2136.000000
22        TN  2030.000000
23        FP   197.000000
24        FN   197.000000


### Testing

In [11]:
import pandas as pd
import numpy as np

# ================= Test File =================
test_file_path = "/home/fidha/Aromatic_Prediction/Auto_trades_opm/idr/data/W_rows_test.csv"

# ================= Load Test Data =================
df_test = pd.read_csv(test_file_path, header=None, sep=r"\s+")
df_test = df_test[~df_test[0].str.contains("X")]

test_sequences = df_test[0].tolist()
test_values = df_test[3].tolist()

# ================= Create Test Labels  =================
test_labels = []

for seq, val in zip(test_sequences, test_values):
    if 'R' not in seq and 'K' not in seq:
        label = 1 if val >= BASE_THRESHOLD else 0
    else:
        threshold = compute_dynamic_threshold(seq)
        label = 1 if val >= threshold else 0
    test_labels.append(label)

# ================= Use Trained Parameters =================
# Make sure this comes from training block
trained_params = params   # result.x from training

# ================= Predict on Test =================
test_raw_scores = [
    model_prediction(seq, trained_params)
    for seq in test_sequences
]

test_preds = [1 if score >= 0.5 else 0 for score in test_raw_scores]

# ================= Confusion Matrix =================
TP = sum(1 for p, l in zip(test_preds, test_labels) if p == 1 and l == 1)
TN = sum(1 for p, l in zip(test_preds, test_labels) if p == 0 and l == 0)
FP = sum(1 for p, l in zip(test_preds, test_labels) if p == 1 and l == 0)
FN = sum(1 for p, l in zip(test_preds, test_labels) if p == 0 and l == 1)

accuracy = (TP + TN) / len(test_labels) * 100

# ================= Print Results =================
print("\n===== TEST SET RESULTS =====")
print("Accuracy:", round(accuracy, 3), "%")
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)


===== TEST SET RESULTS =====
Accuracy: 92.018 %
TP: 581
TN: 468
FP: 45
FN: 46


## Y-centered sequences

### Training

In [32]:
import pandas as pd
import numpy as np
from scipy.optimize import least_squares

# ================= Configuration =================
file_path = "/home/fidha/Aromatic_Prediction/Auto_trades_opm/idr/data/Y_rows_train.csv"
middle_position = 5

# -------- Scale Factor --------
SCALE_FACTOR = 1/(10**-1)**(1/8)

# Thresholding parameters
BASE_THRESHOLD = 40
MIN_THRESHOLD = 9
R_WEIGHT = 15
K_WEIGHT = 10

# Amino acids and groups
aa_list = list("ACDEFGHIKLMNPQRSTVWY")

aa_groups = {
    "group1": list("VLIMFWY"),
    "group2": list("GPCA"),
    "group3": list("STNQH"),
    "group4": list("KR"),
    "group5": list("DE"),
}

aa_to_group = {aa: g for g, aas in aa_groups.items() for aa in aas}


# ================= Sigmoid =================
def sigmoid(Q):
    if Q <= 0:
        return 0.0
    return Q / (1 + Q)


# ================= Model =================
def model_prediction(seq, params):
    product = 1.0
    param_map = {aa: params[i] for i, aa in enumerate(aa_list)}

    for index, aa in enumerate(seq, start=1):
        if index == middle_position or aa not in aa_list:
            continue

        product *= max(1e-12, param_map[aa])

    return sigmoid(product)


# ================= Residuals =================
def residuals(params, sequences, labels):
    return [
        label - model_prediction(seq, params)
        for seq, label in zip(sequences, labels)
    ]


def sum_squared_residuals(params, sequences, labels):
    r = residuals(params, sequences, labels)
    return [x**2 for x in r]


# ================= Threshold  =================
def compute_dynamic_threshold(seq):
    num_r = seq.count('R')
    num_k = seq.count('K')
    return max(BASE_THRESHOLD - R_WEIGHT * num_r - K_WEIGHT * num_k, MIN_THRESHOLD)


# ================= Bounds & Initial Guess (SCALED) =================
def get_bounds_and_guess():

    lower_bounds = []
    upper_bounds = []
    initial_guess = []

    for aa in aa_list:

        group = aa_to_group.get(aa, "group5")

        if group == "group1":
            lb, ub, ig = 0.1, 5.0, 1.87
        elif group == "group2":
            lb, ub, ig = 0.01, 2.0, 0.28
        elif group == "group3":
            lb, ub, ig = 0.01, 1.5, 0.21
        elif group == "group4":
            lb, ub, ig = 0.01, 1.0, 0.32
        else:
            lb, ub, ig = 0.01, 1.0, 0.19

        lower_bounds.append(lb * SCALE_FACTOR)
        upper_bounds.append(ub * SCALE_FACTOR)
        initial_guess.append(ig * SCALE_FACTOR)

    return lower_bounds, upper_bounds, initial_guess


# ================= Load Data =================
df = pd.read_csv(file_path, header=None, sep=r"\s+")
df = df[~df[0].str.contains("X")]

sequences = df[0].tolist()
values = df[3].tolist()


# ================= Label Assignment =================
labels = []

for seq, val in zip(sequences, values):

    if 'R' not in seq and 'K' not in seq:
        label = 1 if val >= BASE_THRESHOLD else 0
    else:
        threshold = compute_dynamic_threshold(seq)
        label = 1 if val >= threshold else 0

    labels.append(label)


# ================= Optimization =================
lower_bounds, upper_bounds, initial_guess = get_bounds_and_guess()

result = least_squares(
    sum_squared_residuals,
    initial_guess,
    bounds=(lower_bounds, upper_bounds),
    args=(sequences, labels),
    method='trf'
)


# ================= Output =================
results_dict = {"AA": aa_list + ["Accuracy", "TP", "TN", "FP", "FN"]}

if result.success:

    params = result.x

    raw_scores = [
        model_prediction(seq, params)
        for seq in sequences
    ]

    preds = [1 if score >= 0.5 else 0 for score in raw_scores]

    TP = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 1)
    TN = sum(1 for p, l in zip(preds, labels) if p == 0 and l == 0)
    FP = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 0)
    FN = sum(1 for p, l in zip(preds, labels) if p == 0 and l == 1)

    accuracy = (TP + TN) / len(labels) * 100

    column_data = list(params) + [accuracy, TP, TN, FP, FN]

else:
    column_data = [np.nan] * (len(aa_list) + 5)


results_dict["Run_full"] = column_data
results_df = pd.DataFrame(results_dict)

print(results_df)

          AA     Run_full
0          A     0.623018
1          C     0.013335
2          D     0.221436
3          E     0.340203
4          F     2.121940
5          G     0.346601
6          H     0.207265
7          I     1.702547
8          K     0.583285
9          L     1.574199
10         M     0.902072
11         N     0.297533
12         P     0.955985
13         Q     0.422065
14         R     1.136965
15         S     0.418921
16         T     0.560483
17         V     1.060686
18         W     1.791123
19         Y     1.273021
20  Accuracy    99.425476
21        TP    14.000000
22        TN  6043.000000
23        FP     6.000000
24        FN    29.000000


### Testing

In [15]:
import pandas as pd
import numpy as np

# ================= Test File =================
test_file_path = "/home/fidha/Aromatic_Prediction/Auto_trades_opm/idr/data/Y_rows_test.csv"

# ================= Load Test Data =================
df_test = pd.read_csv(test_file_path, header=None, sep=r"\s+")
df_test = df_test[~df_test[0].str.contains("X")]

test_sequences = df_test[0].tolist()
test_values = df_test[3].tolist()

# ================= Create Test Labels =================
test_labels = []

for seq, val in zip(test_sequences, test_values):
    if 'R' not in seq and 'K' not in seq:
        label = 1 if val >= BASE_THRESHOLD else 0
    else:
        threshold = compute_dynamic_threshold(seq)
        label = 1 if val >= threshold else 0
    test_labels.append(label)

# ================= Use Trained Parameters =================
# Make sure this comes from training block
trained_params = params   # result.x from training

# ================= Predict on Test =================
test_raw_scores = [
    model_prediction(seq, trained_params)
    for seq in test_sequences
]

test_preds = [1 if score >= 0.5 else 0 for score in test_raw_scores]

# ================= Confusion Matrix =================
TP = sum(1 for p, l in zip(test_preds, test_labels) if p == 1 and l == 1)
TN = sum(1 for p, l in zip(test_preds, test_labels) if p == 0 and l == 0)
FP = sum(1 for p, l in zip(test_preds, test_labels) if p == 1 and l == 0)
FN = sum(1 for p, l in zip(test_preds, test_labels) if p == 0 and l == 1)

accuracy = (TP + TN) / len(test_labels) * 100

# ================= Print Results =================
print("\n===== TEST SET RESULTS =====")
print("Accuracy:", round(accuracy, 3), "%")
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)


===== TEST SET RESULTS =====
Accuracy: 99.738 %
TP: 4
TN: 1516
FP: 0
FN: 4
